In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import time
import joblib
import matplotlib.pyplot as plt
from tvDatafeed import TvDatafeed, Interval
from sklearn.metrics import mean_squared_error, mean_absolute_error
from IPython.display import clear_output

# ==========================================
# 1. ĐỊNH NGHĨA CHÍNH XÁC KIẾN TRÚC MÔ HÌNH
# ==========================================
class HLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # Input -> gates
        self.x2h = nn.Linear(input_size, 5 * hidden_size, bias=True)
        # Hidden -> gates (không bias để đúng paper hơn)
        self.h2h = nn.Linear(hidden_size, 5 * hidden_size, bias=False)

    def forward(self, x, h_prev, c_prev, k_prev):
        # ===== Gates computation =====
        gates = self.x2h(x) + self.h2h(h_prev)
        # Split thành 5 phần
        i, f, o, g, u = gates.chunk(5, dim=1)

        # ===== Activation =====
        i = torch.sigmoid(i)   # input gate
        f = torch.sigmoid(f)   # forget gate
        o = torch.sigmoid(o)   # output gate
        u = torch.sigmoid(u)   # update gate (mới)
        g = torch.tanh(g)      # candidate

        # ===== Hybrid candidate memory =====
        k = u * g + (1 - u) * k_prev
        # ===== Cell state =====
        c = f * c_prev + i * k
        # ===== Hidden state =====
        h = o * torch.tanh(c)
        return h, c, k

class HBLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.fw = HLSTMCell(input_size, hidden_size)
        self.bw = HLSTMCell(input_size, hidden_size)
        self.fc = nn.Linear(2 * hidden_size, 1)

    def forward(self, x):
        batch, seq_len, _ = x.shape
        device = x.device

        # ===== Khởi tạo state =====
        h_f = torch.zeros(batch, self.hidden_size, device=device)
        c_f = torch.zeros_like(h_f)
        k_f = torch.zeros_like(h_f)

        h_b = torch.zeros(batch, self.hidden_size, device=device)
        c_b = torch.zeros_like(h_b)
        k_b = torch.zeros_like(h_b)

        # ===== Forward direction =====
        for t in range(seq_len):
            h_f, c_f, k_f = self.fw(x[:, t, :], h_f, c_f, k_f)

        # ===== Backward direction =====
        for t in reversed(range(seq_len)):
            h_b, c_b, k_b = self.bw(x[:, t, :], h_b, c_b, k_b)

        # ===== Many-to-one Concatenate =====
        out = torch.cat([h_f, h_b], dim=1)  
        # ===== Final Prediction =====
        return self.fc(out)  

# ==========================================
# 2. CÁC HÀM TIỀN XỬ LÝ & ĐÁNH GIÁ
# ==========================================
def preprocess_live_data(df):
    df_clean = df.copy()
    df_clean = df_clean.reset_index()
    df_clean = df_clean.rename(columns={'datetime': 'datetime_col'})
    df_clean = df_clean.dropna()
    df_clean = df_clean.sort_values(by='datetime_col', ascending=True)
    return df_clean

def calculate_ema5(df):
    df_ema = df.copy()
    df_ema['EMA5'] = df_ema['close'].ewm(span=5, adjust=False).mean()
    return df_ema

def inverse_transform_target(scaled_val, scaler, target_idx=3, total_cols=6):
    dummy = np.zeros((1, total_cols))
    dummy[0, target_idx] = scaled_val
    return scaler.inverse_transform(dummy)[0, target_idx]

def mean_absolute_percentage_error(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# ==========================================
# 3. THIẾT LẬP THÔNG SỐ & KHỞI TẠO
# ==========================================
input_size = 6
hidden_size = 32
seq_len = 3
target_col_idx = 3 # close
features_cols = ['open', 'high', 'low', 'close', 'volume', 'EMA5']

device = torch.device("cpu")

# Load Model
model = HBLSTM(input_size, hidden_size).to(device)
model.load_state_dict(torch.load('hblstm_model.pth', map_location=device))
print("✅ Đã load Model thành công!")

# Load Scaler
scaler = joblib.load('scaler.pkl')
print("✅ Đã load Scaler thành công!")

# Khởi tạo Optimizer & Loss để dùng cho Online Learning
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

tv = TvDatafeed()

# ==========================================
# 4. HÀM CHẠY BOT REAL-TIME
# ==========================================
# ==========================================
# HÀM CHẠY BOT THỰC TẾ (CÓ KIỂM TRA PHIÊN GIAO DỊCH)
# ==========================================
def run_realtime_bot(wait_time=60): # Kiểm tra mỗi 1 phút để bắt kịp nến mới nhanh nhất
    actuals = []
    predictions = []
    
    pending_X = None 
    pending_pred_val = None
    last_processed_timestamp = None # Biến quan trọng để kiểm tra nến mới
    
    print("🚀 Bot đang lắng nghe thị trường SSE (000001)...")
    
    try:
        while True:
            # 1. CRAWL DỮ LIỆU
            data = tv.get_hist(symbol='000001', exchange='SSE', interval=Interval.in_15_minute, n_bars=30)
            
            if data is None or len(data) < seq_len:
                time.sleep(60)
                continue
            
            # Lấy nến cuối cùng và kiểm tra thời gian
            current_candle_time = data.index[-1] 
            
            # KIỂM TRA XEM CÓ PHẢI NẾN MỚI KHÔNG
            if last_processed_timestamp is not None and current_candle_time == last_processed_timestamp:
                # Nếu thời gian nến cuối không đổi so với lần trước -> Thị trường đang đóng cửa hoặc chưa hết 15p
                print(f"😴 [{time.strftime('%H:%M:%S')}] Thị trường chưa có nến mới. Đang chờ...")
                time.sleep(wait_time) 
                continue
            
            # Nếu chạy đến đây nghĩa là ĐÃ CÓ NẾN MỚI
            print(f"\n✨ [MỚI] Phát hiện nến mới lúc: {current_candle_time}")
            last_processed_timestamp = current_candle_time # Cập nhật lại thời gian vừa xử lý

            # 2. TIỀN XỬ LÝ
            df_clean = preprocess_live_data(data)
            df_ema = calculate_ema5(df_clean)
            current_actual_close = df_ema['close'].iloc[-1]
            
            # 3. INCREMENTAL LEARNING (Chỉ học khi có nến thực tế mới về)
            if pending_X is not None:
                print(f"📈 So khớp: Dự đoán [{pending_pred_val:.2f}] vs Thực tế [{current_actual_close:.2f}]")
                actuals.append(current_actual_close)
                predictions.append(pending_pred_val)
                
                # Cập nhật trọng số
                model.train()
                optimizer.zero_grad()
                out = model(pending_X)
                
                dummy_target = np.zeros((1, 6))
                dummy_target[0, target_col_idx] = current_actual_close
                scaled_actual = scaler.transform(dummy_target)[0, target_col_idx]
                y_train_tensor = torch.tensor([[scaled_actual]], dtype=torch.float32).to(device)
                
                loss = criterion(out, y_train_tensor)
                loss.backward()
                optimizer.step()
                print(f"✅ Đã cập nhật kiến thức mới. Loss: {loss.item():.6f}")

            # 4. DỰ ĐOÁN CHO 15 PHÚT TỚI
            last_3_candles = df_ema[features_cols].tail(seq_len).values
            scaled_input = scaler.transform(last_3_candles)
            X_tensor = torch.tensor(scaled_input, dtype=torch.float32).unsqueeze(0).to(device)
            
            model.eval()
            with torch.no_grad():
                pred_scaled = model(X_tensor)
                
            pending_X = X_tensor 
            pred_real = inverse_transform_target(pred_scaled.item(), scaler)
            pending_pred_val = pred_real 
            
            print(f"🔮 DỰ BÁO GIÁ NẾN TIẾP THEO: {pred_real:.2f}")
            print(f"⏳ Chờ nến 15 phút tiếp theo xuất hiện...")
            
            # Sau khi xong việc, chờ 1 đoạn ngắn rồi lại kiểm tra xem có nến mới chưa
            time.sleep(wait_time)

    except KeyboardInterrupt:
        print("\n🛑 Dừng Bot. Đang xuất biểu đồ báo cáo...")
        # (Giữ nguyên phần vẽ biểu đồ như code trước của bạn ở đây)

    # ==========================================
    # 5. TÍNH METRICS & VẼ BIỂU ĐỒ SAU KHI KẾT THÚC
    # ==========================================
    if len(actuals) > 0:
        actuals = np.array(actuals)
        predictions = np.array(predictions)
        
        mae = mean_absolute_error(actuals, predictions)
        rmse = np.sqrt(mean_squared_error(actuals, predictions))
        mape = mean_absolute_percentage_error(actuals, predictions)
        
        print("\n" + "="*30)
        print("📊 TỔNG KẾT METRICS DỰ ĐOÁN ONLINE")
        print("="*30)
        print(f"MAE  : {mae:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"MAPE : {mape:.4f} %")
        
        plt.figure(figsize=(10, 5))
        plt.plot(actuals, label='Giá Thực Tế (Actual)', marker='o', color='blue')
        plt.plot(predictions, label='Giá Dự Đoán (Predict)', marker='x', linestyle='--', color='red')
        plt.title('Biểu đồ Thực tế vs Dự đoán (Cơ chế Online Learning)')
        plt.xlabel('Các mốc cập nhật')
        plt.ylabel('Giá Cổ Phiếu')
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.show()
    else:
        print("\n⚠️ Chưa đủ chu kỳ để vẽ biểu đồ (Cần ít nhất chạy 2 lần)")

# ==========================================
# 6. KÍCH HOẠT BOT
# ==========================================
# Để test thử luồng chạy: Để wait_time=5 (Mỗi 5 giây nó check 1 lần)
# Sau khi thấy nó in ra dòng chữ "DỰ BÁO...", hãy tự tay bấm nút STOP (Interrupt) 
# trên Jupyter để xem nó in biểu đồ ra không nhé!
run_realtime_bot(wait_time=5)

✅ Đã load Model thành công!
✅ Đã load Scaler thành công!


you are using nologin method, data you access may be limited


🚀 Bot đang lắng nghe thị trường SSE (000001)...

✨ [MỚI] Phát hiện nến mới lúc: 2026-04-15 14:00:00
🔮 DỰ BÁO GIÁ NẾN TIẾP THEO: 4028.85
⏳ Chờ nến 15 phút tiếp theo xuất hiện...
😴 [14:39:11] Thị trường chưa có nến mới. Đang chờ...
😴 [14:39:17] Thị trường chưa có nến mới. Đang chờ...
😴 [14:39:23] Thị trường chưa có nến mới. Đang chờ...
😴 [14:39:28] Thị trường chưa có nến mới. Đang chờ...
😴 [14:39:35] Thị trường chưa có nến mới. Đang chờ...
😴 [14:39:40] Thị trường chưa có nến mới. Đang chờ...
